In [ ]:
import os
import re
import sys
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass
if not IN_COLAB and os.path.isdir("/content"):
    IN_COLAB = True


def _pip_install_requirements(req: Path, *, skip_torch: bool) -> None:
    if not req.is_file():
        return
    if skip_torch:
        kept = []
        for line in req.read_text().splitlines():
            s = line.split("#", 1)[0].strip()
            if not s:
                continue
            if re.match(r"(?i)^torch\b", s):
                continue
            kept.append(line)
        tmp = Path("/tmp/stk_mat2011_requirements_no_torch.txt")
        tmp.write_text("\n".join(kept) + "\n")
        target = tmp
        print("Colab: pip installing requirements (torch line skipped)")
    else:
        target = req
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(target)], check=False)


REPO_URL = "https://github.com/egil10/stk-mat2011.git"
if IN_COLAB:
    REPO_DIR = Path("/content/stk-mat2011")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
    _pip_install_requirements(REPO_DIR / "requirements.txt", skip_torch=True)
    _nb = REPO_DIR / "code" / "notebooks"
    if _nb.is_dir():
        os.chdir(str(_nb))
    print("Working directory:", os.getcwd())
else:
    REPO_DIR = Path.cwd().resolve().parents[1]
    _pip_install_requirements(REPO_DIR / "requirements.txt", skip_torch=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=False)

In [ ]:
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SYMBOL = "eurusd"

MODEL_MAX_POINTS = 200_000
MODEL_STRIDE_MANUAL = None
CLIP_Q = (0.001, 0.999)
DROP_ZERO_RETURNS = True

SESSION_HOURS = 3.0
AR_ORDER = 1
K_CANDIDATES = (2, 3)

TRAIN_FRAC, VAL_FRAC = 0.70, 0.85
PLOT_TEST_TAIL = 4_000

SWITCHING_AR = True
SWITCHING_VARIANCE = True


def clip_open_close_hours(df: pd.DataFrame, hours: float) -> pd.DataFrame:
    if "date" not in df.columns:
        df = df.copy()
        df["date"] = df["datetime"].dt.date.astype(str)
    pieces = []
    for _, g in df.groupby("date", sort=False):
        g = g.sort_values("datetime")
        t0, t1 = g["datetime"].iloc[0], g["datetime"].iloc[-1]
        span_lo = t0 + pd.Timedelta(hours=hours)
        span_hi = t1 - pd.Timedelta(hours=hours)
        m = (g["datetime"] <= span_lo) | (g["datetime"] >= span_hi)
        pieces.append(g.loc[m])
    out = pd.concat(pieces, ignore_index=True)
    return out.sort_values("datetime").reset_index(drop=True)


def fit_msar(
    y: np.ndarray,
    k_regimes: int,
    *,
    order: int = 1,
    search_reps: int = 10,
    maxiter: int = 200,
):
    y = np.asarray(y, dtype=float).ravel()
    mod = MarkovAutoregression(
        y,
        k_regimes=k_regimes,
        order=order,
        switching_ar=SWITCHING_AR,
        switching_variance=SWITCHING_VARIANCE,
    )
    res = mod.fit(search_reps=search_reps, maxiter=maxiter, disp=False)
    return res

In [ ]:
try:
    from google.colab import drive
    _colab = True
except Exception:
    _colab = False

if _colab:
    drive.mount("/content/drive", force_remount=True)
    OUT_BASES = [
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data"),
    ]
else:
    OUT_BASES = [Path("../data/processed_mega"), Path("../data/processed")]

patterns = [
    f"*{SYMBOL}*master*.parquet",
    f"*mega_master*.parquet",
    "*master_returns*.parquet",
    "*master*.parquet",
]

candidates = []
for base in OUT_BASES:
    if not base.exists():
        continue
    for pat in patterns:
        for p in base.glob(pat):
            if "daily_counts" in p.name:
                continue
            candidates.append((p.stat().st_mtime, p))

if not candidates:
    raise FileNotFoundError("No master parquet found. Set paths or run hw1.")

_, data_path = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
print("Using master file:", data_path)

df = pd.read_parquet(data_path)
if "returns" not in df.columns or "datetime" not in df.columns:
    raise ValueError(f"Need datetime + returns in {data_path}")

df = df.dropna(subset=["returns", "datetime"]).copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

if DROP_ZERO_RETURNS:
    df = df[df["returns"] != 0].copy()

if CLIP_Q is not None:
    lo, hi = df["returns"].quantile(CLIP_Q[0]), df["returns"].quantile(CLIP_Q[1])
    df["returns"] = df["returns"].clip(lo, hi)

if "date" not in df.columns:
    df["date"] = df["datetime"].dt.date.astype(str)

if MODEL_STRIDE_MANUAL is not None:
    stride = int(MODEL_STRIDE_MANUAL)
else:
    stride = 1
    if len(df) > MODEL_MAX_POINTS:
        stride = int(np.ceil(len(df) / MODEL_MAX_POINTS))

print(f"Original points: {len(df):,} | Using stride={stride}")
if stride > 1:
    df = df.iloc[::stride].copy().reset_index(drop=True)

df_b = df.reset_index(drop=True)

n = len(df_b)
i1, i2 = int(TRAIN_FRAC * n), int(VAL_FRAC * n)
df_tr_b = df_b.iloc[:i1].copy()
df_va_b = df_b.iloc[i1:i2].copy()
df_te_b = df_b.iloc[i2:].copy()
print("train:", len(df_tr_b), "val:", len(df_va_b), "test:", len(df_te_b))
print("train range:", df_tr_b["datetime"].min(), "->", df_tr_b["datetime"].max())

display(df_b.head())

In [ ]:
df_raw = df.copy()

def apply_stride(d):
    if MODEL_STRIDE_MANUAL is not None:
        st = int(MODEL_STRIDE_MANUAL)
    else:
        st = 1
        if len(d) > MODEL_MAX_POINTS:
            st = int(np.ceil(len(d) / MODEL_MAX_POINTS))
    if st > 1:
        d = d.iloc[::st].copy().reset_index(drop=True)
    return d, st


df_a = clip_open_close_hours(df_raw, SESSION_HOURS)
df_b = df_raw

st_a, st_b = None, None
df_a, st_a = apply_stride(df_a)
df_b, st_b = apply_stride(df_b)
print(f"Protocol A: {len(df_a):,} points (stride={st_a})")
print(f"Protocol B: {len(df_b):,} points (stride={st_b})")


def split_df(d):
    n = len(d)
    i1, i2 = int(TRAIN_FRAC * n), int(VAL_FRAC * n)
    return d.iloc[:i1], d.iloc[i1:i2], d.iloc[i2:]


df_tr_a, df_va_a, df_te_a = split_df(df_a.reset_index(drop=True))
df_tr_b, df_va_b, df_te_b = split_df(df_b.reset_index(drop=True))

In [ ]:
try:
    from google.colab import drive
    _colab = True
except Exception:
    _colab = False

if _colab:
    drive.mount("/content/drive", force_remount=True)
    OUT_BASES = [
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data"),
    ]
else:
    OUT_BASES = [Path("../data/processed_mega"), Path("../data/processed")]

patterns = [
    f"*{SYMBOL}*master*.parquet",
    f"*mega_master*.parquet",
    "*master_returns*.parquet",
    "*master*.parquet",
]

candidates = []
for base in OUT_BASES:
    if not base.exists():
        continue
    for pat in patterns:
        for p in base.glob(pat):
            if "daily_counts" in p.name:
                continue
            candidates.append((p.stat().st_mtime, p))

if not candidates:
    raise FileNotFoundError("No master parquet found. Set paths or run hw1.")

_, data_path = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
print("Using master file:", data_path)

df = pd.read_parquet(data_path)
if "returns" not in df.columns or "datetime" not in df.columns:
    raise ValueError(f"Need datetime + returns in {data_path}")

df = df.dropna(subset=["returns", "datetime"]).copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

if DROP_ZERO_RETURNS:
    df = df[df["returns"] != 0].copy()

if CLIP_Q is not None:
    lo, hi = df["returns"].quantile(CLIP_Q[0]), df["returns"].quantile(CLIP_Q[1])
    df["returns"] = df["returns"].clip(lo, hi)

if "date" not in df.columns:
    df["date"] = df["datetime"].dt.date.astype(str)

df_raw = df.copy()


def apply_stride(d):
    if MODEL_STRIDE_MANUAL is not None:
        st = int(MODEL_STRIDE_MANUAL)
    else:
        st = 1
        if len(d) > MODEL_MAX_POINTS:
            st = int(np.ceil(len(d) / MODEL_MAX_POINTS))
    print(f"  n={len(d):,} -> stride={st}")
    if st > 1:
        d = d.iloc[::st].copy().reset_index(drop=True)
    return d


print("Protocol A (clip):")
df_a = clip_open_close_hours(df_raw, SESSION_HOURS)
df_a = apply_stride(df_a)

print("Protocol B (full):")
df_b = apply_stride(df_raw.copy())


def split_df(d):
    d = d.reset_index(drop=True)
    n = len(d)
    i1, i2 = int(TRAIN_FRAC * n), int(VAL_FRAC * n)
    return d.iloc[:i1], d.iloc[i1:i2], d.iloc[i2:]


df_tr_a, df_va_a, df_te_a = split_df(df_a)
df_tr_b, df_va_b, df_te_b = split_df(df_b)

print("A — train:", len(df_tr_a), "val:", len(df_va_a), "test:", len(df_te_a))
print("B — train:", len(df_tr_b), "val:", len(df_va_b), "test:", len(df_te_b))
display(df_b.head())

In [ ]:
rows = []
fitted = {}

for tag, tr in [("A", df_tr_a), ("B", df_tr_b)]:
    y_tr = tr["returns"].astype(float).values
    for k in K_CANDIDATES:
        key = (tag, k)
        print("fitting", key, "n=", len(y_tr))
        res = fit_msar(y_tr, k, order=AR_ORDER)
        fitted[key] = res
        rows.append(
            {
                "protocol": tag,
                "K": k,
                "n_train": len(y_tr),
                "llf": float(res.llf),
                "bic": float(res.bic),
                "aic": float(res.aic),
            }
        )

summary = pd.DataFrame(rows).sort_values("bic").reset_index(drop=True)
display(summary)
best = (summary.iloc[0]["protocol"], int(summary.iloc[0]["K"]))
print("Best by BIC (train-only):", best)

In [ ]:
tag, k_best = best
if tag == "A":
    df_full = pd.concat([df_tr_a, df_va_a, df_te_a], ignore_index=True)
else:
    df_full = pd.concat([df_tr_b, df_va_b, df_te_b], ignore_index=True)

y_full = df_full["returns"].astype(float).values
res_plot = fit_msar(y_full, k_best, order=AR_ORDER)

order = AR_ORDER
sp = res_plot.smoothed_marginal_probabilities
t_plot = df_full["datetime"].iloc[order : order + len(sp)].values

n_te = len(df_te_a) if tag == "A" else len(df_te_b)
start = max(order, len(df_full) - max(PLOT_TEST_TAIL, n_te))
# align test tail
mask = np.arange(len(sp)) >= (start - order)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_plot[mask], y_full[start:], color="black", lw=0.5, label="returns")
axes[0].set_ylabel("return")
axes[0].legend(loc="upper right")

for j in range(k_best):
    axes[1].plot(t_plot[mask], sp[mask, j], lw=1.0, label=f"P(regime {j})")
axes[1].set_ylabel("smoothed prob")
axes[1].set_xlabel("time")
axes[1].legend(loc="upper right")
axes[0].set_title(f"MS-AR(1) protocol {tag}, K={k_best} (refit on train+val+test)")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(8, 3))
plot_acf(df_tr_b["returns"].dropna(), lags=30, ax=ax, title="ACF train returns (Protocol B)")
plt.tight_layout()
plt.show()